# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     |  David Farias Rivera |
| **Fecha**      |  11/10/2026 |
| **Expediente** |  754381 | 

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion). 

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$. 

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing 
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then 
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds. 

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

In [1]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

### Datos y validación cruzada

In [2]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print('Medidas:', iris.feature_names)
print('Especies:', iris.target_names)
print('Forma de X:', X_iris.shape)

cv_iris = KFold(n_splits=10, shuffle=True, random_state=15)

Medidas: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Especies: ['setosa' 'versicolor' 'virginica']
Forma de X: (150, 4)


### SVC: kernel lineal

In [3]:
modelo_linear_iris = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", probability=True, random_state=15)
)

auc_linear_iris = cross_val_score(
    modelo_linear_iris,
    X_iris,
    y_iris,
    cv=cv_iris,
    scoring="roc_auc_ovr",
    error_score="raise"
)

print("AUC linear:", auc_linear_iris.mean())

AUC linear: 0.9992424242424243


### SVC: kernel polinomial

In [4]:
modelo_poly_iris = make_pipeline(
    StandardScaler(),
    SVC(kernel="poly", probability=True, random_state=15)
)

auc_poly_iris = cross_val_score(
    modelo_poly_iris,
    X_iris,
    y_iris,
    cv=cv_iris,
    scoring="roc_auc_ovr",
    error_score="raise"
)

print("AUC poly:", auc_poly_iris.mean())

AUC poly: 0.9955308641975309


### SVC: kernel RBF

In [5]:
modelo_rbf_iris = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", probability=True, random_state=15)
)

auc_rbf_iris = cross_val_score(
    modelo_rbf_iris,
    X_iris,
    y_iris,
    cv=cv_iris,
    scoring="roc_auc_ovr",
    error_score="raise"
)

print("AUC RBF:", auc_rbf_iris.mean())

AUC RBF: 1.0


### SVC: kernel sigmoide

In [6]:
modelo_sigmoid_iris = make_pipeline(
    StandardScaler(),
    SVC(kernel="sigmoid", probability=True, random_state=15)
)

auc_sigmoid_iris = cross_val_score(
    modelo_sigmoid_iris,
    X_iris,
    y_iris,
    cv=cv_iris,
    scoring="roc_auc_ovr",
    error_score="raise"
)

print("AUC sigmoid:", auc_sigmoid_iris.mean())

AUC sigmoid: 0.9785773115773114


### Regresión logística

In [7]:
modelo_logistico_iris = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

auc_logistico_iris = cross_val_score(
    modelo_logistico_iris,
    X_iris,
    y_iris,
    cv=cv_iris,
    scoring="roc_auc_ovr",
    error_score="raise"
)

print("AUC regresión logística:", auc_logistico_iris.mean())

AUC regresión logística: 0.9970606060606061


### Resultados de Iris

In [8]:
tabla_iris = pd.DataFrame({
    "Modelo": ["SVC linear", "SVC poly", "SVC rbf", "SVC sigmoid", "Regresión logística"],
    "AUC promedio": [
        auc_linear_iris.mean(),
        auc_poly_iris.mean(),
        auc_rbf_iris.mean(),
        auc_sigmoid_iris.mean(),
        auc_logistico_iris.mean()
    ],
    "Desviación": [
        auc_linear_iris.std(),
        auc_poly_iris.std(),
        auc_rbf_iris.std(),
        auc_sigmoid_iris.std(),
        auc_logistico_iris.std()
    ]
})

display(tabla_iris.sort_values("AUC promedio", ascending=False).round(4))

,Modelo,AUC promedio,Desviación
2,SVC rbf,1.0000,0.0000
0,SVC linear,0.9992,0.0023
4,Regresión logística,0.9971,0.0069
1,SVC poly,0.9955,0.0099
3,SVC sigmoid,0.9786,0.0307


**Comparación de Iris.** Con los mismos 10 folds y AUC multiclase para los cinco modelos, SVC con kernel `rbf` obtuvo el AUC promedio más alto (1.0000). Le siguieron `linear` (0.9992), regresión logística (0.9971), `poly` (0.9955) y `sigmoid` (0.9786). Los resultados de `rbf`, `linear` y regresión logística son muy cercanos. AUC mide la capacidad de distinguir clases a partir de puntuaciones o probabilidades; no es el porcentaje de predicciones correctas.

## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

### Datos y validación cruzada

In [9]:
df = pd.read_csv(
    "/Users/david/Documents/Codigod/ITESO/01-ITESO/Lab 3/Default.csv"
)

X_default = df[["student", "balance", "income"]].copy()
X_default["student"] = (X_default["student"] == "Yes").astype(int)
y_default = (df["default"] == "Yes").astype(int)

print(X_default.head())
print("Casos por clase:")
print(y_default.value_counts())

cv_default = KFold(n_splits=10, shuffle=True, random_state=15)

   student      balance        income
0        0   729.526495  44361.625074
1        1   817.180407  12106.134700
2        0  1073.549164  31767.138947
3        0   529.250605  35704.493935
4        0   785.655883  38463.495879
Casos por clase:
default
0    9667
1     333
Name: count, dtype: int64


### SVC: kernel lineal

In [10]:
modelo_linear_default = make_pipeline(
    StandardScaler(),
    SVC(kernel="linear")
)

auc_linear_default = cross_val_score(
    modelo_linear_default,
    X_default,
    y_default,
    cv=cv_default,
    scoring="roc_auc",
    error_score="raise"
)

print("AUC linear:", auc_linear_default.mean())

AUC linear: 0.9282898980303538


### SVC: kernel polinomial

In [11]:
modelo_poly_default = make_pipeline(
    StandardScaler(),
    SVC(kernel="poly")
)

auc_poly_default = cross_val_score(
    modelo_poly_default,
    X_default,
    y_default,
    cv=cv_default,
    scoring="roc_auc",
    error_score="raise"
)

print("AUC poly:", auc_poly_default.mean())

AUC poly: 0.874517812976439


### SVC: kernel RBF

In [12]:
modelo_rbf_default = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf")
)

auc_rbf_default = cross_val_score(
    modelo_rbf_default,
    X_default,
    y_default,
    cv=cv_default,
    scoring="roc_auc",
    error_score="raise"
)

print("AUC RBF:", auc_rbf_default.mean())

AUC RBF: 0.8371956948009679


### SVC: kernel sigmoide

In [13]:
modelo_sigmoid_default = make_pipeline(
    StandardScaler(),
    SVC(kernel="sigmoid")
)

auc_sigmoid_default = cross_val_score(
    modelo_sigmoid_default,
    X_default,
    y_default,
    cv=cv_default,
    scoring="roc_auc",
    error_score="raise"
)

print("AUC sigmoid:", auc_sigmoid_default.mean())

AUC sigmoid: 0.7359683872475475


### Regresión logística

In [14]:
modelo_logistico_default = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

auc_logistico_default = cross_val_score(
    modelo_logistico_default,
    X_default,
    y_default,
    cv=cv_default,
    scoring="roc_auc",
    error_score="raise"
)

print("AUC regresión logística:", auc_logistico_default.mean())

AUC regresión logística: 0.9489033532854932


### Resultados de Default

In [15]:
tabla_default = pd.DataFrame({
    "Modelo": ["SVC linear", "SVC poly", "SVC rbf", "SVC sigmoid", "Regresión logística"],
    "AUC promedio": [
        auc_linear_default.mean(),
        auc_poly_default.mean(),
        auc_rbf_default.mean(),
        auc_sigmoid_default.mean(),
        auc_logistico_default.mean()
    ],
    "Desviación": [
        auc_linear_default.std(),
        auc_poly_default.std(),
        auc_rbf_default.std(),
        auc_sigmoid_default.std(),
        auc_logistico_default.std()
    ]
})

display(tabla_default.sort_values("AUC promedio", ascending=False).round(4))

,Modelo,AUC promedio,Desviación
4,Regresión logística,0.9489,0.0089
0,SVC linear,0.9283,0.0216
1,SVC poly,0.8745,0.0208
2,SVC rbf,0.8372,0.0491
3,SVC sigmoid,0.7360,0.0370


**Comparación de Default.** La regresión logística alcanzó el mayor AUC promedio (0.9489). Entre los SVC, el kernel `linear` fue el mejor (0.9283), seguido de `poly` (0.8745), `rbf` (0.8372) y `sigmoid` (0.7360). La variable objetivo `default` tiene 333 casos `Yes` y 9,667 casos `No`; por eso AUC resume la capacidad de distinguir ambos grupos sin usar únicamente el porcentaje de aciertos. Estos valores corresponden a los 10 folds y las configuraciones usadas arriba.

# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result